### 1. Create a single dataframe with the concatenation of all input csv files, adding a column called "country"

In [67]:
import pandas as pd
import chardet  #  for encoding

def detect_encoding(file_path): # This function guess the type of coding like  utf-8,ISO-8859-1...
    with open(file_path, 'rb') as f:
        raw_data = f.read(100000)
        result = chardet.detect(raw_data)
        return result['encoding']

files = {
    "DE": r"C:\Users\User\Desktop\DEvideos.csv",  # r means "raw" otherwise I had to change the direction of slash \
    "CA": r"C:\Users\User\Desktop\CAvideos.csv",
    "FR": r"C:\Users\User\Desktop\FRvideos.csv",
    "GB": r"C:\Users\User\Desktop\GBvideos.csv",
    "IN": r"C:\Users\User\Desktop\INvideos.csv", # error start from here until RU file
    "JP": r"C:\Users\User\Desktop\JPvideos.csv",
    "KR": r"C:\Users\User\Desktop\KRvideos.csv",
    "MX": r"C:\Users\User\Desktop\MXvideos.csv",
    "RU": r"C:\Users\User\Desktop\RUvideos.csv",
    "US": r"C:\Users\User\Desktop\USvideos.csv"
}

dfs = []      # created the empty list so I will save the dataframe 

for country, path in files.items():
    enc = detect_encoding(path)           
    print(f"{country}: {enc}")            
    df = pd.read_csv(path, encoding=enc, encoding_errors='replace') # if can not read a byte change it with box insteaf of error
    df["country"] = country
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True) # it delete previous index
final_df.head()

DE: Windows-1254
CA: Windows-1254
FR: Windows-1254
GB: Windows-1254
IN: Windows-1254
JP: None
KR: None
MX: Windows-1254
RU: utf-8
US: Windows-1254


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,LgVi6y5QIjM,17.14.11,Sing zu Ende! | Gesangseinlagen vom Feinsten |...,inscope21,24,2017-11-13T17:08:49.000Z,"inscope21|""sing zu ende""|""gesangseinlagen""|""ge...",252786,35885,230,1539,https://i.ytimg.com/vi/LgVi6y5QIjM/default.jpg,False,False,False,Heute gibt es mal wieder ein neues Format... w...,DE
1,Bayt7uQith4,17.14.11,Kinder ferngesteuert im Kiosk! Erwachsene abzo...,LUKE! Die Woche und ich,23,2017-11-12T22:30:01.000Z,"Kinder|""ferngesteuert""|""Kinder ferngesteuert""|...",797196,53576,302,1278,https://i.ytimg.com/vi/Bayt7uQith4/default.jpg,False,False,False,Kinder ferngesteuert! Kinder lassen sich sooo ...,DE
2,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97190,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",DE
3,AHtypnRk7JE,17.14.11,Das Fermi-Paradoxon,100SekundenPhysik,27,2017-11-12T15:00:01.000Z,"Physik|""Wissenschaft""|""Technik""|""Science-Ficti...",380247,31821,458,1955,https://i.ytimg.com/vi/AHtypnRk7JE/default.jpg,False,False,False,â–ºAlle Videos: http://bit.ly/1fa7Tw3\n\n\nâœš...,DE
4,ZJ9We4bjcg0,17.14.11,18 SONGS mit Kelly MissesVlog (Sing-off),rezo,24,2017-11-12T13:10:36.000Z,"kelly|""missesvlog""|""kelly song""|""bausa""|""bausa...",822213,100684,2467,10244,https://i.ytimg.com/vi/ZJ9We4bjcg0/default.jpg,False,False,False,18 Song Mashup Ã¼ber den (verÃ¤nderten) Beat v...,DE


### 2. Extract all videos that have no tag.

In [68]:
unknown_placeholders = ["Unknown", "none", "missing"]# fillna("") → replaces missing (NaN) values with empty string 
                                                     # because strstrip dont work with NaN
                                                       # str.strip() → removes spaces from both ends of the string
                                                        # == "" → selects rows where tags are empty after cleaning
no_tags_df = final_df[
    (final_df["tags"].fillna("").str.strip() == "") |
     (final_df["tags"].str.lower().str.strip().isin(unknown_placeholders))
] 
no_tags_df.head()  # there is no empty tag
#print(final_df["tags"].isna().sum())

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country


### 3. For each channel, determine the total number of views

In [69]:
invalid_values = ["unknown", "missing"]            # Remove rows where channel title is NaN, empty string, or "Unknown"
                                                   # not NaN
clean_df = final_df[                               # not empty string
    final_df["channel_title"].notna() &            # not labeled as Unknown,missing etc. case insensitive
    (final_df["channel_title"].str.strip() != "") &  # ! means not equal 
    (~final_df["channel_title"].str.strip().str.lower().isin(invalid_values))].copy()        
                         
             
clean_df["views"] = pd.to_numeric(clean_df["views"], errors="coerce") # If there are string number like "1000" convert to real
                                                                      # number or make it NaN so sum won't get error

#clean_df["channel_title"] = clean_df["channel_title"].str.strip().str.lower()  # If want to make case-insensitive

channel_views = clean_df.groupby("channel_title")["views"].sum()  # compute total views per valid channel

sorted_views = channel_views.sort_values(ascending=False) # If I want to find to most views


#channel_name = "TechWorld"  # If I want to find spesific channel name I can use this code
#if channel_name in channel_views.index:
#    print(channel_views.loc[channel_name])
#else:
#    print("Not find.")



channel_views.head()

channel_title
! 세상에 무슨일이                 3942977
!!8時だよ面白ネタ大集合                50207
!BTS・TWICE まとめ                7310
!Los amorosos ViralesÂ¡       6069
!t Live                     240038
Name: views, dtype: int64

### 4. Save all rows with disabled comments and disabled ratings, or that have "video_error_or_removed" in a new dataframe called "excluded", and remove those rows from the original dataframe.

In [70]:
condition = ( (final_df['comments_disabled'] == True) & (final_df['ratings_disabled'] == True) ) | \
            (final_df['video_error_or_removed'] == True) # If there is NaN then that row not exsit in this condition but there isn't

excluded = final_df[condition].copy() # create another dataframe

final_df = final_df[~condition].copy() # delete these rows from original dataframe


print(f"Excluded row number: {len(excluded)}")
print(f"remain rows number: {len(final_df)}")

Excluded row number: 2620
remain rows number: 373322


In [71]:
final_df.isna().sum() # see the NaN values

video_id                      0
trending_date                 0
title                         0
channel_title                 0
category_id                   0
publish_time                  0
tags                          0
views                         0
likes                         0
dislikes                      0
comment_count                 0
thumbnail_link                0
comments_disabled             0
ratings_disabled              0
video_error_or_removed        0
description               19165
country                       0
dtype: int64

### 5. Add a "like_ratio" column storing the ratio between the number of "likes" and of "dislikes"


In [72]:
final_df['likes'] = pd.to_numeric(final_df['likes'], errors='coerce') 
final_df['dislikes'] = pd.to_numeric(final_df['dislikes'], errors='coerce') # If there is a not number they will be NaN
                                                                            # and string convert to numeric

final_df['like_ratio'] = final_df['likes'] / final_df['dislikes'].replace(0, pd.NA)# If dislike is 0 the result be NaN


final_df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio
0,LgVi6y5QIjM,17.14.11,Sing zu Ende! | Gesangseinlagen vom Feinsten |...,inscope21,24,2017-11-13T17:08:49.000Z,"inscope21|""sing zu ende""|""gesangseinlagen""|""ge...",252786,35885,230,1539,https://i.ytimg.com/vi/LgVi6y5QIjM/default.jpg,False,False,False,Heute gibt es mal wieder ein neues Format... w...,DE,156.021739
1,Bayt7uQith4,17.14.11,Kinder ferngesteuert im Kiosk! Erwachsene abzo...,LUKE! Die Woche und ich,23,2017-11-12T22:30:01.000Z,"Kinder|""ferngesteuert""|""Kinder ferngesteuert""|...",797196,53576,302,1278,https://i.ytimg.com/vi/Bayt7uQith4/default.jpg,False,False,False,Kinder ferngesteuert! Kinder lassen sich sooo ...,DE,177.403974
2,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97190,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",DE,15.813537
3,AHtypnRk7JE,17.14.11,Das Fermi-Paradoxon,100SekundenPhysik,27,2017-11-12T15:00:01.000Z,"Physik|""Wissenschaft""|""Technik""|""Science-Ficti...",380247,31821,458,1955,https://i.ytimg.com/vi/AHtypnRk7JE/default.jpg,False,False,False,â–ºAlle Videos: http://bit.ly/1fa7Tw3\n\n\nâœš...,DE,69.478166
4,ZJ9We4bjcg0,17.14.11,18 SONGS mit Kelly MissesVlog (Sing-off),rezo,24,2017-11-12T13:10:36.000Z,"kelly|""missesvlog""|""kelly song""|""bausa""|""bausa...",822213,100684,2467,10244,https://i.ytimg.com/vi/ZJ9We4bjcg0/default.jpg,False,False,False,18 Song Mashup Ã¼ber den (verÃ¤nderten) Beat v...,DE,40.812323


### 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)


In [73]:
final_df['publish_time'] = pd.to_datetime(final_df['publish_time'])# converting string to date-time so I can work on it

intervals = final_df['publish_time'].dt.floor('10min')  # it round down the time like 14.07>14.00

final_df['publish_10min_interval'] = (      # I created a new feature and this feature is string
    intervals.dt.strftime('%H:%M') + '-' +
    (intervals + pd.Timedelta(minutes=10)).dt.strftime('%H:%M'))

final_df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_10min_interval
0,LgVi6y5QIjM,17.14.11,Sing zu Ende! | Gesangseinlagen vom Feinsten |...,inscope21,24,2017-11-13 17:08:49+00:00,"inscope21|""sing zu ende""|""gesangseinlagen""|""ge...",252786,35885,230,1539,https://i.ytimg.com/vi/LgVi6y5QIjM/default.jpg,False,False,False,Heute gibt es mal wieder ein neues Format... w...,DE,156.021739,17:00-17:10
1,Bayt7uQith4,17.14.11,Kinder ferngesteuert im Kiosk! Erwachsene abzo...,LUKE! Die Woche und ich,23,2017-11-12 22:30:01+00:00,"Kinder|""ferngesteuert""|""Kinder ferngesteuert""|...",797196,53576,302,1278,https://i.ytimg.com/vi/Bayt7uQith4/default.jpg,False,False,False,Kinder ferngesteuert! Kinder lassen sich sooo ...,DE,177.403974,22:30-22:40
2,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13 07:30:00+00:00,"last week tonight trump presidency|""last week ...",2418783,97190,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",DE,15.813537,07:30-07:40
3,AHtypnRk7JE,17.14.11,Das Fermi-Paradoxon,100SekundenPhysik,27,2017-11-12 15:00:01+00:00,"Physik|""Wissenschaft""|""Technik""|""Science-Ficti...",380247,31821,458,1955,https://i.ytimg.com/vi/AHtypnRk7JE/default.jpg,False,False,False,â–ºAlle Videos: http://bit.ly/1fa7Tw3\n\n\nâœš...,DE,69.478166,15:00-15:10
4,ZJ9We4bjcg0,17.14.11,18 SONGS mit Kelly MissesVlog (Sing-off),rezo,24,2017-11-12 13:10:36+00:00,"kelly|""missesvlog""|""kelly song""|""bausa""|""bausa...",822213,100684,2467,10244,https://i.ytimg.com/vi/ZJ9We4bjcg0/default.jpg,False,False,False,18 Song Mashup Ã¼ber den (verÃ¤nderten) Beat v...,DE,40.812323,13:10-13:20


### 7. For each interval, determine the number of videos, average number of likes and of dislikes.


In [74]:
interval_stats = final_df.groupby('publish_10min_interval').agg(   # calculation without NaN values.
    video_count=('publish_10min_interval', 'count'),               # count only non-nulls
    avg_likes=('likes', 'mean'),                                   # mean ignores the NaN values
    avg_dislikes=('dislikes', 'mean')
).reset_index()


interval_stats['start_time'] = pd.to_datetime(    # for sorting it convert string  to date-time new feature
    interval_stats['publish_10min_interval'].str.split('-').str[0], format='%H:%M') # In here take only the first time section

interval_stats = interval_stats.sort_values('start_time').drop('start_time', axis=1)
#interval_stats.sort_values('avg_likes', ascending=False)  # sorting  likes
#interval_stats.sort_values('avg_dislikes', ascending=False) #sorting dislikes
#interval_stats.sort_values('video_count', ascending=False) # sorting video count


print(interval_stats.head())

  publish_10min_interval  video_count     avg_likes  avg_dislikes
0            00:00-00:10         2897  61288.115637   3808.149465
1            00:10-00:20         1509  22748.138502   1449.836315
2            00:20-00:30         1241  21378.280419   1072.344883
3            00:30-00:40         1614  36853.560719    955.890954
4            00:40-00:50         1269  42198.623325   1909.301812


### 8. For each tag, determine the number of videos 
#### Notice that tags contains a string with several tags

In [75]:
final_df['tags'] = final_df['tags'].fillna('').str.lower().replace('[none]', '') # NaN values became empty strings so split works
                                                                     # "none" values became empty strings so split works


final_df['tag_list'] = final_df['tags'].str.split('|') # convert to tag list, use '|(pipe)' because youtube uses it


tags_exploded = final_df.explode('tag_list') # exploaded tags list


tags_exploded = tags_exploded[tags_exploded['tag_list'].str.strip() != ''] # delete the empty tags



tag_counts = (tags_exploded.drop_duplicates(['video_id', 'tag_list']).groupby('tag_list').size().reset_index(name='video_count')
    .sort_values('video_count', ascending=False).reset_index(drop=True))     # calculate the video number per each  tag without duplicates
                      

 
print(tag_counts.head(20))  # we can see the most uses 20 tags

 

           tag_list  video_count
0            "2018"         5563
1           "funny"         4647
2            "news"         4052
3          "comedy"         3930
4              "tv"         3027
5            "2017"         2576
6           "video"         2332
7            "show"         2229
8             "diy"         2068
9           "music"         2017
10     "television"         1996
11          "путин"         1862
12        "youtube"         1824
13           "vlog"         1810
14           "live"         1793
15  "entertainment"         1744
16         "россия"         1734
17       "politics"         1585
18        "новости"         1549
19       "football"         1519


### 9. Find the tags with the largest number of videos

In [76]:
print(tag_counts.sort_values('video_count', ascending=False).head(20)) # already done previous question

           tag_list  video_count
0            "2018"         5563
1           "funny"         4647
2            "news"         4052
3          "comedy"         3930
4              "tv"         3027
5            "2017"         2576
6           "video"         2332
7            "show"         2229
8             "diy"         2068
9           "music"         2017
10     "television"         1996
11          "путин"         1862
12        "youtube"         1824
13           "vlog"         1810
14           "live"         1793
15  "entertainment"         1744
16         "россия"         1734
17       "politics"         1585
18        "новости"         1549
19       "football"         1519


### 10. For each (tag, country) pair, compute average ratio likes/dislikes

In [77]:
tag_country_avg_ratio = tags_exploded.groupby(['tag_list', 'country'])['like_ratio'].mean().reset_index()
tag_country_avg_ratio.columns = ['tag_list', 'country', 'avg_like_ratio'] # giving name to coloumns

tag_country_avg_ratio = tag_country_avg_ratio.sort_values('avg_like_ratio',ascending=False).reset_index(drop=True) # sorting
 
print(tag_country_avg_ratio.head(20))

 

                    tag_list country avg_like_ratio
0                      "att"      RU        11688.0
1           "at&t originals"      RU        11688.0
2                 "directtv"      RU        11688.0
3                "direct tv"      RU        11688.0
4                  "u-verse"      RU        11688.0
5                       at&t      RU        11688.0
6         "audience network"      RU        11688.0
7            "at&t audience"      RU        11688.0
8    "at&t audience network"      RU        11688.0
9                "originals"      RU    5844.403005
10                     "스타쉽"      RU        3550.75
11                "mosnta x"      RU        3550.75
12                   "드라마라마"      RU        3550.75
13               "dramarama"      RU        3550.75
14         "dramarama dance"      RU        3550.75
15  "dramarama choreography"      RU        3550.75
16                "드라마라마 안무"      RU        3550.75
17                   "남자 안무"      RU        3550.75
18          

### 11. For each (trending_date, country) pair, the video with the largest number of views# 

In [78]:
final_df['views'] = pd.to_numeric(final_df['views'], errors='coerce') # convert to numeric and if there are unnumeric, it will be NaN

idx = final_df.groupby(['trending_date', 'country'])['views'].idxmax() # find the index has max views ( if there are same take first one)

top_videos = final_df.loc[idx, ['trending_date', 'views', 'video_id', 'country']] \
                     .reset_index(drop=True) # choose the index find before

top_videos = top_videos.sort_values('views', ascending=False).reset_index(drop=True) # sorting 

top_videos.head(20)

,trending_date,views,video_id,country
0,18.07.04,424538912,_I_D_8Z4sJE,GB
1,18.06.04,413586699,_I_D_8Z4sJE,GB
2,18.05.04,402650804,_I_D_8Z4sJE,GB
3,18.04.04,392036878,_I_D_8Z4sJE,GB
4,18.03.04,382401497,_I_D_8Z4sJE,GB
5,18.02.04,372399338,_I_D_8Z4sJE,GB
6,18.01.04,362111555,_I_D_8Z4sJE,GB
7,18.31.03,349987176,_I_D_8Z4sJE,GB
8,18.30.03,339629489,_I_D_8Z4sJE,GB
9,18.18.05,337621571,9jI-z9QN6g8,GB


### 12. Divide trending_date into three columns: year, month, day

In [87]:
final_df['trending_date'] = final_df['trending_date'].astype(str).str.strip()

final_df['trending_date'] = pd.to_datetime(final_df['trending_date'], ) # convert to date-time( if can not became NaN)
                                                                                    #because to use dt.
final_df['year'] = final_df['trending_date'].dt.year
final_df['month'] = final_df['trending_date'].dt.month
final_df['day'] = final_df['trending_date'].dt.day
final_df.head(5)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_10min_interval,tag_list,year,month,day
0,LgVi6y5QIjM,2017-11-14,Sing zu Ende! | Gesangseinlagen vom Feinsten |...,inscope21,24,2017-11-13 17:08:49+00:00,"inscope21|""sing zu ende""|""gesangseinlagen""|""ge...",252786,35885,230,...,False,False,Heute gibt es mal wieder ein neues Format... w...,DE,156.021739,17:00-17:10,"[inscope21, ""sing zu ende"", ""gesangseinlagen"",...",2017,11,14
1,Bayt7uQith4,2017-11-14,Kinder ferngesteuert im Kiosk! Erwachsene abzo...,LUKE! Die Woche und ich,23,2017-11-12 22:30:01+00:00,"kinder|""ferngesteuert""|""kinder ferngesteuert""|...",797196,53576,302,...,False,False,Kinder ferngesteuert! Kinder lassen sich sooo ...,DE,177.403974,22:30-22:40,"[kinder, ""ferngesteuert"", ""kinder ferngesteuer...",2017,11,14
2,1ZAPwfrtAFY,2017-11-14,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13 07:30:00+00:00,"last week tonight trump presidency|""last week ...",2418783,97190,6146,...,False,False,"One year after the presidential election, John...",DE,15.813537,07:30-07:40,"[last week tonight trump presidency, ""last wee...",2017,11,14
3,AHtypnRk7JE,2017-11-14,Das Fermi-Paradoxon,100SekundenPhysik,27,2017-11-12 15:00:01+00:00,"physik|""wissenschaft""|""technik""|""science-ficti...",380247,31821,458,...,False,False,â–ºAlle Videos: http://bit.ly/1fa7Tw3\n\n\nâœš...,DE,69.478166,15:00-15:10,"[physik, ""wissenschaft"", ""technik"", ""science-f...",2017,11,14
4,ZJ9We4bjcg0,2017-11-14,18 SONGS mit Kelly MissesVlog (Sing-off),rezo,24,2017-11-12 13:10:36+00:00,"kelly|""missesvlog""|""kelly song""|""bausa""|""bausa...",822213,100684,2467,...,False,False,18 Song Mashup Ã¼ber den (verÃ¤nderten) Beat v...,DE,40.812323,13:10-13:20,"[kelly, ""missesvlog"", ""kelly song"", ""bausa"", ""...",2017,11,14


### 13. For each (month, country) pair, the video with the largest number of views

In [89]:
final_df['views'] = pd.to_numeric(final_df['views'], errors='coerce') # convert views to numeric (if there are unnumerci will be NaN)

idx = final_df.groupby(['month', 'country'])['views'].idxmax() # finding the largest's index

top_videos_month_country = final_df.loc[idx].reset_index(drop=True)
top_videos_month_country = top_videos_month_country.sort_values(by='views', ascending=False).reset_index(drop=True) # sorting 
top_videos_month_country = top_videos_month_country[['month', 'country', 'video_id', 'views']]

print(top_videos_month_country)



    month country     video_id      views
0       4      GB  _I_D_8Z4sJE  424538912
1       3      GB  _I_D_8Z4sJE  349987176
2       5      GB  9jI-z9QN6g8  337621571
3       2      GB  wfWkmURBNv8  280125114
4       6      GB  VYOjWnS4cMY  259721696
..    ...     ...          ...        ...
72     11      RU  kTlv5_Bs8aw   20565795
73      6      RU  aJOTlE1K90k   17303018
74      2      RU  xpVfcZ0ZcFM   17059675
75      6      JP  TIE92mUvSsw   15969920
76      2      JP  wbSwFU6tY1c   14816949

[77 rows x 4 columns]


### 14. Read all json files with the video categories


In [97]:
import json
import glob
import os
import pandas as pd

categories = []  # empty list to collect all categories

for file in glob.glob(r"C:\Users\User\Desktop\trendingYT\*_category_id.json"):
    
    country = os.path.basename(file).split("_")[0]

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

        for item in data['items']:
            categories.append({
                'category_id': int(item['id']),
                'category_title': item['snippet']['title'],
                'country': country
            })

categories_df = pd.DataFrame(categories)

print(categories_df.head(5))

   category_id    category_title country
0            1  Film & Animation      CA
1            2  Autos & Vehicles      CA
2           10             Music      CA
3           15    Pets & Animals      CA
4           17            Sports      CA


###  15. For each country, determine how many videos have a category that is not assignable.


In [109]:
final_df['category_id'] = pd.to_numeric(final_df['category_id'], errors='coerce') # convert to numeric( if can not became NaN)

not_assignable_count = (     
    final_df[final_df['category_id'].isna()]    # find the all videos has no category
    .groupby('country')                         # make them group
    .size()
    .reset_index(name='not_assignable_videos')
)

not_assignable_count = not_assignable_count.sort_values(by='not_assignable_videos',ascending=False)# sorting it
not_assignable_count = not_assignable_count.reset_index(drop=True) # reset the index
not_assignable_count.head()

,country,not_assignable_videos
